# sparse_rebound_release_target_revision_v1

Notebook-first continuation from `conservative_rebound_release_gate_v1`. The prior diagnostic found that current fold-local state-objective targets contain **zero** effective rebound lift because downtrend risk caps win before rebound floors. It also found a conservative SOXX/SOXL spread-oscillation release gate with heldout precision `1.000`, recall `0.227`, event FPR `0.000`, and post-event effective false-positive rows `0`.

This notebook applies that sparse release gate to the target table as a **target-semantics revision only**:

- preserve the original risk cap in `risk_target_position_original`;
- lift `risk_target_position` only where the selected release gate, rebound label, downtrend state, and material lift agree;
- re-run target-lift, target-scope, and BC label-distribution readiness;
- do not run PPO or allow runtime promotion from this target-readiness notebook.

Governance fields:

- **Canonical baseline**: `v19reg_v22_budget_usage_bull_guard_v1`.
- **Direct comparison baseline**: `fold_local_joint_reward_targets_v1`; previous diagnostic `conservative_rebound_release_gate_v1`.
- **Role**: `diagnose`.
- **Changed layer**: `model_selection`, `reward`, `feature`.
- **Target window**: fold-local train rows only; `2022` and later eval rows remain excluded by scope audit.
- **Accepted trade-off**: sparse recall is acceptable if the label table becomes trainable without false-positive release amplification.
- **Failure mode**: if label distribution degenerates or scope/lift gates fail, do not run BC/PPO.

External references retained from the previous gate design:

- Qlib: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- FinRL: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- Focal Loss for sparse events: https://arxiv.org/abs/1708.02002
- Prioritized Experience Replay for sparse/high-error transitions: https://arxiv.org/abs/1511.05952
- Probability of Backtest Overfitting / CSCV: https://doi.org/10.21314/JCF.2016.322


In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from src.soxl_quant.app.state_objective_bc import build_state_objective_bc_dataset_from_frame
from src.soxl_quant.app.state_objective_training_targets import (
    StateObjectiveBCReadinessConfig,
    apply_state_objective_sparse_release_gate,
    audit_state_objective_target_lift,
    audit_state_objective_target_scope,
    score_state_objective_bc_readiness,
    summarize_state_objective_bc_readiness,
    summarize_state_objective_target_lift_audit,
    summarize_state_objective_target_scope_audit,
    summarize_state_objective_training_targets,
)
from src.soxl_quant.config import load_profile

EXPERIMENT = 'sparse_rebound_release_target_revision_v1'
CARD = ROOT / 'runtime/model_selection/card' / EXPERIMENT
SOURCE_CARD = ROOT / 'runtime/model_selection/card/fold_local_joint_reward_targets_v1'
GATE_CARD = ROOT / 'runtime/model_selection/card/conservative_rebound_release_gate_v1'
TARGETS_PATH = SOURCE_CARD / 'fold_local_joint_targets.csv'
RELEASE_SCORES_PATH = GATE_CARD / 'post_event_release_row_scores.csv'
GATE_SELECTION_PATH = GATE_CARD / 'selection.json'
CREATED_AT = datetime.now(timezone.utc).isoformat()
CARD.mkdir(parents=True, exist_ok=True)

TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0

folds = [
    {
        'name': 'wf_2022_train',
        'source_window': 'strict_oos_train_source',
        'train_start': '2020-01-02',
        'train_end': '2021-12-31',
        'eval_start': '2022-01-03',
        'eval_end': '2022-12-30',
    },
    {
        'name': 'wf_2023_2024_train',
        'source_window': 'strict_oos_train_source',
        'train_start': '2020-01-02',
        'train_end': '2022-12-30',
        'eval_start': '2023-01-03',
        'eval_end': '2024-12-31',
    },
]


def json_default(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if pd.isna(value):
        return None
    raise TypeError(f'Object of type {type(value).__name__} is not JSON serializable')


def write_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, default=json_default) + '\n', encoding='utf-8')


def action_entropy(actions: np.ndarray) -> float:
    if len(actions) == 0:
        return 0.0
    counts = pd.Series(actions).value_counts(normalize=True)
    return float(-(counts * np.log(counts)).sum())


def label_distribution_rows(target_table: pd.DataFrame, *, label: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    feature_frame = pd.read_pickle(ROOT / 'data/soxl_profile_balanced_rule_residual_v22.pkl')
    profiles = ['risk_budget_usage_v1', 'balanced', 'balanced_rule_residual_v22']
    rows = []
    fold_rows = []
    for profile_name in profiles:
        profile = load_profile(profile_name)
        dataset = build_state_objective_bc_dataset_from_frame(
            feature_frame,
            targets=target_table,
            profile=profile,
            sample_size=0,
        )
        actions = dataset.actions
        if len(actions):
            counts = pd.Series(actions).value_counts()
            dominant_action = int(counts.index[0])
            dominant_share = float(counts.iloc[0] / len(actions))
            unique_actions = int(pd.Series(actions).nunique())
        else:
            dominant_action = -1
            dominant_share = 1.0
            unique_actions = 0
        rows.append({
            'target_label': label,
            'profile': profile_name,
            'samples': int(len(actions)),
            'unique_actions': unique_actions,
            'dominant_action': dominant_action,
            'dominant_action_share': dominant_share,
            'action_entropy': action_entropy(actions),
            'obs_dim': int(dataset.obs.shape[1]) if len(dataset.obs.shape) > 1 else 0,
            'rebound_samples': int(dataset.metadata['is_rebound_cluster'].sum()) if 'is_rebound_cluster' in dataset.metadata else 0,
            'downtrend_samples': int(dataset.metadata['is_downtrend_state'].sum()) if 'is_downtrend_state' in dataset.metadata else 0,
            'release_gate_samples': int(dataset.metadata.get('release_gate_state', pd.Series(False, index=dataset.metadata.index)).sum()) if not dataset.metadata.empty else 0,
        })
        for fold_name, subset in target_table.groupby('fold'):
            fold_dataset = build_state_objective_bc_dataset_from_frame(
                feature_frame,
                targets=subset,
                profile=profile,
                sample_size=0,
            )
            fold_actions = fold_dataset.actions
            if len(fold_actions):
                fold_counts = pd.Series(fold_actions).value_counts()
                fold_dominant_action = int(fold_counts.index[0])
                fold_dominant_share = float(fold_counts.iloc[0] / len(fold_actions))
                fold_unique_actions = int(pd.Series(fold_actions).nunique())
            else:
                fold_dominant_action = -1
                fold_dominant_share = 1.0
                fold_unique_actions = 0
            fold_rows.append({
                'target_label': label,
                'fold': str(fold_name),
                'profile': profile_name,
                'samples': int(len(fold_actions)),
                'unique_actions': fold_unique_actions,
                'dominant_action': fold_dominant_action,
                'dominant_action_share': fold_dominant_share,
                'action_entropy': action_entropy(fold_actions),
                'release_gate_samples': int(fold_dataset.metadata.get('release_gate_state', pd.Series(False, index=fold_dataset.metadata.index)).sum()) if not fold_dataset.metadata.empty else 0,
            })
    return pd.DataFrame(rows), pd.DataFrame(fold_rows)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
targets = pd.read_csv(TARGETS_PATH, parse_dates=['datetime'])
release_scores = pd.read_csv(RELEASE_SCORES_PATH, parse_dates=['datetime'])
gate_selection = json.loads(GATE_SELECTION_PATH.read_text())

selected_feature_set = str(gate_selection['selected_feature_set'])
selected_threshold = float(gate_selection['selected_threshold'])

revised_targets = apply_state_objective_sparse_release_gate(
    targets,
    release_scores,
    feature_set=selected_feature_set,
    threshold=selected_threshold,
    min_lift=0.01,
    require_rebound_label=True,
)
revised_targets.to_csv(CARD / 'sparse_release_revised_targets.csv', index=False)

release_summary = (
    revised_targets.groupby('fold')
    .agg(
        rows=('datetime', 'count'),
        rebound_rows=('is_rebound_cluster', 'sum'),
        downtrend_rows=('is_downtrend_state', 'sum'),
        release_gate_predicted_rows=('release_gate_predicted', 'sum'),
        release_gate_rows=('release_gate_state', 'sum'),
        mean_release_gate_lift=('release_gate_lift', lambda s: float(pd.to_numeric(s, errors='coerce').where(revised_targets.loc[s.index, 'release_gate_state']).mean()) if revised_targets.loc[s.index, 'release_gate_state'].any() else 0.0),
        max_release_gate_lift=('release_gate_lift', 'max'),
    )
    .reset_index()
)
release_summary.to_csv(CARD / 'release_gate_target_summary.csv', index=False)
release_summary


/Users/rziiiii/.openclaw/workspace/src/soxl_quant/app/state_objective_training_targets.py:1034: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  result["release_gate_predicted"] = result["_date_key"].map(predicted_by_date).fillna(False).astype(bool)


,fold,rows,rebound_rows,downtrend_rows,release_gate_predicted_rows,release_gate_rows,mean_release_gate_lift,max_release_gate_lift
0,wf_2022_train,505,66,234,40,40,0.372998,0.556976
1,wf_2023_2024_train,756,66,478,40,40,0.372998,0.556976


In [3]:
target_lift_audit = audit_state_objective_target_lift(
    revised_targets,
    group_cols=('fold',),
    baseline_risk_position_col='risk_target_position_original',
)
target_lift_summary = summarize_state_objective_target_lift_audit(target_lift_audit)
target_lift_audit.to_csv(CARD / 'target_lift_audit.csv', index=False)
write_json(CARD / 'target_lift_summary.json', target_lift_summary)

target_scope_audit = audit_state_objective_target_scope(revised_targets, folds=folds)
target_scope_summary = summarize_state_objective_target_scope_audit(target_scope_audit)
target_scope_audit.to_csv(CARD / 'target_scope_audit.csv', index=False)
write_json(CARD / 'target_scope_summary.json', target_scope_summary)

target_summary = summarize_state_objective_training_targets(revised_targets)
write_json(CARD / 'target_summary.json', target_summary)

display(target_lift_audit)
display(target_scope_audit)
{
    'target_lift_summary': target_lift_summary,
    'target_scope_summary': target_scope_summary,
    'target_summary_rows': target_summary.get('rows'),
}


,fold,rows,rebound_rows,downtrend_rows,rebound_downtrend_rows,current_positive_rebound_lift_rows,candidate_release_positive_lift_rows,mean_current_effective_lift,mean_candidate_release_lift,max_current_effective_lift,max_candidate_release_lift,lift_semantics_gate,release_candidate_gate,target_position_priority,min_lift
0,wf_2022_train,505,66,234,57,40,57,0.226059,0.329555,0.556976,0.556976,True,True,risk,1.000000e-09
1,wf_2023_2024_train,756,66,478,57,40,57,0.226059,0.329555,0.556976,0.556976,True,True,risk,1.000000e-09


,window,target_scope,target_fold,fold,rows,train_overlap_rows,eval_overlap_rows,outside_train_eval_rows,leakage_gate,scope_issue,train_start,train_end,eval_start,eval_end
0,wf_2022_train,fold_local,wf_2022_train,wf_2022_train,505,505,0,0,True,,2020-01-02,2021-12-31,2022-01-03,2022-12-30
1,wf_2023_2024_train,fold_local,wf_2023_2024_train,wf_2023_2024_train,756,756,0,0,True,,2020-01-02,2022-12-30,2023-01-03,2024-12-31


{'target_lift_summary': {'decision': 'state_objective_target_lift_training_ready',
  'training_allowed': True,
  'lift_semantics_gate': True,
  'audited_groups': 2,
  'blocked_groups': 0,
  'rebound_rows': 132,
  'current_positive_rebound_lift_rows': 80,
  'candidate_release_positive_lift_rows': 114,
  'rebound_downtrend_rows': 114,
  'max_candidate_release_lift': 0.5569758652,
  'next_mechanism': 'run_fold_local_weighted_bc_or_ppo'},
 'target_scope_summary': {'decision': 'state_objective_targets_training_ready',
  'training_allowed': True,
  'leakage_gate': True,
  'audited_groups': 2,
  'blocked_groups': 0,
  'eval_overlap_rows': 0,
  'outside_train_eval_rows': 0,
  'scope_issues': [],
  'top_scope_issue': '',
  'next_mechanism': 'run_fold_local_state_objective_training_or_bc'},
 'target_summary_rows': 1261}

In [4]:
base_labels, base_fold_labels = label_distribution_rows(targets, label='base_risk_priority')
revised_labels, revised_fold_labels = label_distribution_rows(revised_targets, label='sparse_release_revised')
profile_label_comparison = pd.concat([base_labels, revised_labels], ignore_index=True)
fold_profile_label_comparison = pd.concat([base_fold_labels, revised_fold_labels], ignore_index=True)

revised_only = revised_labels[revised_labels['target_label'] == 'sparse_release_revised'].drop(columns=['target_label'])
bc_readiness_scores = score_state_objective_bc_readiness(
    revised_only,
    target_scope='fold_local',
    config=StateObjectiveBCReadinessConfig(
        max_dominant_action_share=0.90,
        min_unique_actions=3,
        min_samples=100,
    ),
)
bc_summary = summarize_state_objective_bc_readiness(bc_readiness_scores)

profile_label_comparison.to_csv(CARD / 'profile_label_comparison.csv', index=False)
fold_profile_label_comparison.to_csv(CARD / 'fold_profile_label_comparison.csv', index=False)
bc_readiness_scores.to_csv(CARD / 'bc_readiness_scores.csv', index=False)
write_json(CARD / 'bc_readiness_summary.json', bc_summary)

display(profile_label_comparison)
display(fold_profile_label_comparison)
bc_summary


,target_label,profile,samples,unique_actions,dominant_action,dominant_action_share,action_entropy,obs_dim,rebound_samples,downtrend_samples,release_gate_samples
0,base_risk_priority,risk_budget_usage_v1,755,5,2,0.313907,1.482661,315,66,477,0
1,base_risk_priority,balanced,755,10,0,0.829139,0.801255,79,66,477,0
2,base_risk_priority,balanced_rule_residual_v22,755,5,0,0.923179,0.374393,340,66,477,0
3,sparse_release_revised,risk_budget_usage_v1,755,5,1,0.295364,1.490966,315,66,477,40
4,sparse_release_revised,balanced,755,10,0,0.805298,0.870599,79,66,477,40
5,sparse_release_revised,balanced_rule_residual_v22,755,5,0,0.894040,0.470479,340,66,477,40


,target_label,fold,profile,samples,unique_actions,dominant_action,dominant_action_share,action_entropy,release_gate_samples
0,base_risk_priority,wf_2022_train,risk_budget_usage_v1,505,5,1,0.277228,1.493996,0
1,base_risk_priority,wf_2023_2024_train,risk_budget_usage_v1,755,5,2,0.313907,1.482661,0
2,base_risk_priority,wf_2022_train,balanced,505,10,0,0.760396,1.044271,0
3,base_risk_priority,wf_2023_2024_train,balanced,755,10,0,0.829139,0.801255,0
4,base_risk_priority,wf_2022_train,balanced_rule_residual_v22,505,5,0,0.885149,0.511214,0
5,base_risk_priority,wf_2023_2024_train,balanced_rule_residual_v22,755,5,0,0.923179,0.374393,0
6,sparse_release_revised,wf_2022_train,risk_budget_usage_v1,505,5,4,0.304950,1.487183,40
7,sparse_release_revised,wf_2023_2024_train,risk_budget_usage_v1,755,5,1,0.295364,1.490966,40
8,sparse_release_revised,wf_2022_train,balanced,505,10,0,0.724752,1.135339,40
9,sparse_release_revised,wf_2023_2024_train,balanced,755,10,0,0.805298,0.870599,40


{'decision': 'state_objective_bc_ready_for_training',
 'selected_profile': 'risk_budget_usage_v1',
 'selected_target_scope': 'fold_local',
 'selected_bc_readiness_gate': True,
 'selected_label_distribution_gate': True,
 'selected_fold_local_scope_gate': True,
 'selected_dominant_action_share': 0.295364238410596,
 'selected_unique_actions': 5,
 'selected_samples': 755,
 'selected_bc_readiness_loss': 0.0,
 'next_mechanism': 'run_fold_local_bc_warm_start_then_walkforward_ppo'}

In [5]:
lift_gate = bool(target_lift_summary.get('training_allowed', False))
scope_gate = bool(target_scope_summary.get('training_allowed', False))
bc_gate = bool(bc_summary.get('selected_bc_readiness_gate', False))
release_gate_rows = int(revised_targets['release_gate_state'].sum())

if lift_gate and scope_gate and bc_gate and release_gate_rows > 0:
    decision = 'sparse_release_targets_bc_screen_ready_no_training'
    next_mechanism = 'run_fold_local_bc_warmstart_with_release_slice_audit_then_walkforward_ppo'
elif not lift_gate:
    decision = 'sparse_release_targets_lift_gate_failed'
    next_mechanism = 'revise_release_gate_or_target_semantics_before_bc'
elif not scope_gate:
    decision = 'sparse_release_targets_scope_blocked'
    next_mechanism = 'regenerate_fold_local_targets_without_eval_overlap'
elif not bc_gate:
    decision = 'sparse_release_targets_label_distribution_blocked'
    next_mechanism = 'change_action_surface_or_target_mapping_before_bc'
else:
    decision = 'sparse_release_targets_no_release_rows'
    next_mechanism = 'discover_more_stable_release_rows'

selected_profile = str(bc_summary.get('selected_profile', ''))
selected_row = bc_readiness_scores[bc_readiness_scores['profile'].astype(str) == selected_profile].iloc[0].to_dict() if selected_profile else {}

selection = {
    'experiment': EXPERIMENT,
    'created_at': CREATED_AT,
    'decision': decision,
    'next_mechanism': next_mechanism,
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'fold_local_joint_reward_targets_v1',
    'prior_diagnostic': str(GATE_CARD.relative_to(ROOT)),
    'role': 'diagnose',
    'changed_layer': ['model_selection', 'reward', 'feature'],
    'source_targets': str(TARGETS_PATH.relative_to(ROOT)),
    'source_release_scores': str(RELEASE_SCORES_PATH.relative_to(ROOT)),
    'selected_release_feature_set': selected_feature_set,
    'selected_release_threshold': selected_threshold,
    'release_gate_rows': release_gate_rows,
    'release_gate_rows_by_fold': release_summary.set_index('fold')['release_gate_rows'].to_dict(),
    'target_lift_gate': lift_gate,
    'target_scope_gate': scope_gate,
    'bc_readiness_gate': bc_gate,
    'bc_training_allowed_next': bool(lift_gate and scope_gate and bc_gate and release_gate_rows > 0),
    'training_allowed': False,
    'promotion_allowed': False,
    'promotion_blocker': 'This notebook only revises target semantics and screens BC labels; fixed-split and walk-forward backtests are still required.',
    'target_rows': int(target_summary.get('rows', 0)),
    'target_rebound_rows': int(target_summary.get('rebound_rows', 0)),
    'target_downtrend_rows': int(target_summary.get('downtrend_rows', 0)),
    'current_positive_rebound_lift_rows': int(target_lift_summary.get('current_positive_rebound_lift_rows', 0)),
    'candidate_release_positive_lift_rows': int(target_lift_summary.get('candidate_release_positive_lift_rows', 0)),
    'scope_eval_overlap_rows': int(target_scope_summary.get('eval_overlap_rows', 0)),
    'selected_profile_for_next_training': selected_profile,
    'selected_unique_actions': int(selected_row.get('unique_actions', 0)),
    'selected_dominant_action_share': float(selected_row.get('dominant_action_share', 1.0)),
    'selected_samples': int(selected_row.get('samples', 0)),
    'selected_release_gate_samples': int(selected_row.get('release_gate_samples', 0)),
    'external_sources': [
        'https://github.com/microsoft/qlib',
        'https://arxiv.org/abs/2009.11189',
        'https://github.com/AI4Finance-Foundation/FinRL',
        'https://arxiv.org/abs/2011.09607',
        'https://arxiv.org/abs/1708.02002',
        'https://arxiv.org/abs/1511.05952',
        'https://doi.org/10.21314/JCF.2016.322',
    ],
    'acceptance_gate_for_next_run': {
        'fold_local_target_scope_required': True,
        'eval_overlap_rows_required': 0,
        'target_lift_gate_required': True,
        'label_distribution_gate_required': True,
        'release_slice_accuracy_required_after_bc': True,
        'fixed_split_required_after_bc': True,
        'walkforward_required_after_bc': True,
        'relative_return_min_pp': TARGET_RELATIVE_RETURN,
        'max_dd_max_pct': TARGET_MAX_DD,
        'no_runtime_promotion_from_target_readiness_only': True,
    },
    'sidecars': [
        'sparse_release_revised_targets.csv',
        'release_gate_target_summary.csv',
        'target_lift_audit.csv',
        'target_lift_summary.json',
        'target_scope_audit.csv',
        'target_scope_summary.json',
        'target_summary.json',
        'profile_label_comparison.csv',
        'fold_profile_label_comparison.csv',
        'bc_readiness_scores.csv',
        'bc_readiness_summary.json',
        'selection.json',
        'summary.json',
    ],
}

summary = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'bc_training_allowed_next': selection['bc_training_allowed_next'],
    'selected_profile_for_next_training': selected_profile,
    'conclusion': [
        f"Sparse release revision created {release_gate_rows} gated target rows across fold-local train targets.",
        f"Target-lift audit now has {selection['current_positive_rebound_lift_rows']} current positive rebound-lift rows versus {selection['candidate_release_positive_lift_rows']} release-candidate rows.",
        f"Scope audit has {selection['scope_eval_overlap_rows']} eval-overlap rows and gate={scope_gate}.",
        f"BC readiness selected {selected_profile} with {selection['selected_unique_actions']} actions, dominant share {selection['selected_dominant_action_share']:.3f}, and {selection['selected_release_gate_samples']} release-gate samples.",
        'Conclusion: target semantics are ready for a BC warm-start screen if the next run audits release-slice fit and still requires fixed-split plus walk-forward +30pp / <=15% MaxDD gates.',
    ],
    'selection': selection,
}

write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)
for line in summary['conclusion']:
    display(Markdown(f'- {line}'))
selection


- Sparse release revision created 80 gated target rows across fold-local train targets.

- Target-lift audit now has 80 current positive rebound-lift rows versus 114 release-candidate rows.

- Scope audit has 0 eval-overlap rows and gate=True.

- BC readiness selected risk_budget_usage_v1 with 5 actions, dominant share 0.295, and 40 release-gate samples.

- Conclusion: target semantics are ready for a BC warm-start screen if the next run audits release-slice fit and still requires fixed-split plus walk-forward +30pp / <=15% MaxDD gates.

{'experiment': 'sparse_rebound_release_target_revision_v1',
 'created_at': '2026-05-25T17:57:30.932126+00:00',
 'decision': 'sparse_release_targets_bc_screen_ready_no_training',
 'next_mechanism': 'run_fold_local_bc_warmstart_with_release_slice_audit_then_walkforward_ppo',
 'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
 'direct_comparison_baseline': 'fold_local_joint_reward_targets_v1',
 'prior_diagnostic': 'runtime/model_selection/card/conservative_rebound_release_gate_v1',
 'role': 'diagnose',
 'changed_layer': ['model_selection', 'reward', 'feature'],
 'source_targets': 'runtime/model_selection/card/fold_local_joint_reward_targets_v1/fold_local_joint_targets.csv',
 'source_release_scores': 'runtime/model_selection/card/conservative_rebound_release_gate_v1/post_event_release_row_scores.csv',
 'selected_release_feature_set': 'soxx_spread_oscillation',
 'selected_release_threshold': 0.75,
 'release_gate_rows': 80,
 'release_gate_rows_by_fold': {'wf_2022_train': 40, 'wf